# CNN nâng cấp — Banana Ripeness
Giữ dataset, nhãn, resize/padding 256×256 và thuật toán chia nhóm seed42 của baseline. Không thêm augmentation online.

**Cách chạy:** bật GPU Kaggle; chọn EXPERIMENT ở cell cấu hình; dùng manifest_used.csv của baseline nếu đã upload, hoặc tái tạo bằng thuật toán cũ trên đúng dataset. Chạy từ đầu sau mỗi lần đổi cấu hình.

- `improved`: CNN 4 block, mỗi block 2 Conv + BatchNorm + ReLU, MaxPool và SpatialDropout; Dense + Dropout; Adam và ReduceLROnPlateau.
- `baseline_control`: kiến trúc baseline cũ, nhưng cùng learning rate, số epoch tối đa và scheduler với improved. Dùng để phân biệt tác động kiến trúc với tác động lịch huấn luyện.
- `baseline_original`: kiến trúc và cấu hình huấn luyện cũ (30 epochs, Adam 0.001, patience6, không scheduler).

Mọi chế độ bật deterministic ops để giảm biến thiên khi chạy lại trong cùng môi trường. Không đảm bảo giống kết quả cũ hoặc giống giữa các phiên bản GPU/TensorFlow. SPLIT_SEED luôn42; TRAIN_SEED có thể đổi để kiểm tra độ ổn định mà không đổi split.

Đây là các cấu hình thí nghiệm, chưa có kết quả huấn luyện và không bảo đảm cải thiện. Test mặc định tắt. Không sử dụng trọng số pretrained.


In [ ]:
import os, json, random, importlib.util, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SPLIT_SEED = 42
TRAIN_SEED = 42
SEED = SPLIT_SEED  # tương thích thuật toán chia nhóm baseline
random.seed(TRAIN_SEED)
np.random.seed(TRAIN_SEED)
tf.keras.utils.set_random_seed(TRAIN_SEED)
tf.config.experimental.enable_op_determinism()
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Cấu hình và chia dữ liệu
DATA_ROOT là thư mục chứa trực tiếp Green, Semi-ripe, Ripe, Overripe; để None để tự tìm trong Kaggle Input. Nếu upload ZIP chưa giải nén, giải nén vào một thư mục Kaggle rồi đặt DATA_ROOT tới thư mục chứa 4 lớp.

MANIFEST_CSV tùy chọn: dùng lại manifest đã chốt, cột path,label,split. Đường dẫn tương đối tính từ DATA_ROOT. Notebook kiểm tra nhóm tên nguồn không bị chia sang các split khác nhau.


In [ ]:
DATA_ROOT = None  # Hoặc đường dẫn thư mục chứa trực tiếp 4 lớp trong /kaggle/input
MANIFEST_CSV = None  # Ví dụ: "/kaggle/input/my-clean-banana/manifest_clean.csv"
CLASS_NAMES = ["Green", "Semi-ripe", "Ripe", "Overripe"]
SPLIT_RATIOS = (0.70, 0.15, 0.15)
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EXPERIMENT = "improved"  # improved / baseline_control / baseline_original
assert EXPERIMENT in {"improved", "baseline_control", "baseline_original"}
LEARNING_RATE = 0.001 if EXPERIMENT == "baseline_original" else 0.0003
EPOCHS = 30 if EXPERIMENT == "baseline_original" else 60
PATIENCE = 6 if EXPERIMENT == "baseline_original" else 12
USE_LR_SCHEDULER = EXPERIMENT != "baseline_original"
RUN_TEST = False  # Chỉ bật sau khi đã chốt toàn bộ mô hình/cấu hình.

from datetime import datetime, timezone
base_output = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT = base_output / ("cnn_" + EXPERIMENT + "_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
OUTPUT.mkdir(parents=True, exist_ok=False)

if DATA_ROOT is None:
    candidates = sorted({p.parent.resolve() for p in Path("/kaggle/input").rglob("Green")
                         if p.is_dir() and all((p.parent / label).is_dir() for label in CLASS_NAMES)})
    if len(candidates) != 1:
        raise ValueError(f"Đặt DATA_ROOT tới thư mục chứa đủ 4 lớp. Tìm thấy: {list(map(str, candidates))}")
    DATA_ROOT = candidates[0]
DATA_ROOT = Path(DATA_ROOT).resolve()
assert DATA_ROOT.is_dir(), f"Không tìm thấy: {DATA_ROOT}"
print("Data:", DATA_ROOT)
print("Output:", OUTPUT)

In [ ]:
label_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}

def source_id(path):
    # Không thêm nhãn vào group: cùng tên nguồn ở hai lớp cũng cần kiểm tra.
    return Path(path).name.split(".rf.")[0]

def assign_group_splits(frame):
    frame = frame.copy()
    frame["source_id"] = frame.path.map(source_id)
    conflicts = frame.groupby("source_id").label.nunique()
    assert not (conflicts > 1).any(), "Có tên nguồn xuất hiện ở nhiều lớp; cần rà soát trước khi chia."
    rng = np.random.default_rng(SEED)
    mapping = {}
    for label in CLASS_NAMES:
        groups = np.array(sorted(frame.loc[frame.label == label, "source_id"].unique()))
        assert len(groups) >= 7, f"Quá ít nhóm ảnh nguồn cho lớp {label}"
        rng.shuffle(groups)
        n_train = int(len(groups) * SPLIT_RATIOS[0])
        n_valid = int(len(groups) * SPLIT_RATIOS[1])
        for split, part in zip(["train", "valid", "test"],
                               [groups[:n_train], groups[n_train:n_train+n_valid], groups[n_train+n_valid:]]):
            mapping.update({str(g): split for g in part})
    frame["split"] = frame.source_id.map(mapping)
    return frame

if MANIFEST_CSV:
    df = pd.read_csv(MANIFEST_CSV)
    assert {"path", "label", "split"}.issubset(df.columns), "CSV cần path,label,split"
    df = df[["path", "label", "split"]].copy()
    assert not df.isna().any().any(), "Manifest có giá trị thiếu"
    df["path"] = df.path.map(lambda p: str((DATA_ROOT / str(p)).resolve()))
    df["source_id"] = df.path.map(source_id)
else:
    extensions = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    rows = [{"path": str(p.resolve()), "label": label}
            for label in CLASS_NAMES for p in sorted((DATA_ROOT / label).rglob("*"))
            if p.is_file() and p.suffix.lower() in extensions]
    df = pd.DataFrame(rows, columns=["path", "label"])
    assert len(df), "Không tìm thấy ảnh"
    df = assign_group_splits(df)
assert len(df), "Manifest rỗng"
assert set(df.label) == set(CLASS_NAMES), f"Nhãn không khớp: {sorted(df.label.unique())}"
assert set(df.split) == {"train", "valid", "test"}, "Cần đủ train/valid/test"
assert not df.path.duplicated().any(), "Đường dẫn ảnh bị lặp"
assert (df.groupby("source_id").split.nunique() == 1).all(), "Rò rỉ nhóm tên nguồn giữa các split"
assert (df.groupby("source_id").label.nunique() == 1).all(), "Cùng tên nguồn nhưng khác nhãn"
missing = [p for p in df.path if not Path(p).is_file()]
assert not missing, f"Không tìm thấy ảnh: {missing[:3]}"
counts = pd.crosstab(df.label, df.split).reindex(index=CLASS_NAMES, columns=["train", "valid", "test"], fill_value=0)
assert (counts > 0).all().all(), "Mỗi split cần đủ 4 lớp"
df["label_id"] = df.label.map(label_to_id)
df = df.sort_values(["split", "label_id", "path"]).reset_index(drop=True)
print("Số ảnh theo lớp/split:")
display(counts)
print("Số nhóm nguồn:")
display(df.groupby(["label", "split"]).source_id.nunique().unstack().reindex(CLASS_NAMES))
print("Thứ tự nhãn:", label_to_id)
print("Chia theo tên nguồn; chưa kiểm tra gần trùng giữa các tên khác nhau.")
# Lưu đường dẫn tương đối để cả nhóm có thể dùng lại trên Kaggle.
manifest = df.copy()
manifest["path"] = manifest.path.map(lambda p: Path(p).relative_to(DATA_ROOT).as_posix())
manifest.to_csv(OUTPUT / "manifest_used.csv", index=False)
(OUTPUT / "class_names.json").write_text(json.dumps(CLASS_NAMES, ensure_ascii=False, indent=2), encoding="utf-8")
config = dict(experiment=EXPERIMENT, split_seed=SPLIT_SEED, train_seed=TRAIN_SEED, deterministic_ops=True,
              scheduler=USE_LR_SCHEDULER, lr_factor=0.5, lr_patience=4, min_lr=1e-6, seed=SEED, image_size=list(IMAGE_SIZE), batch_size=BATCH_SIZE,
              learning_rate=LEARNING_RATE, epochs=EPOCHS, patience=PATIENCE,
              class_names=CLASS_NAMES, preprocessing="EXIF RGB; aspect-preserving bilinear resize; black padding; divide by 255",
              checkpoint_monitor="val_loss", online_augmentation=False, dataset_already_augmented=True,
              split_ratios=list(SPLIT_RATIOS), split_method="stratified by class; grouped by filename before .rf.",
              data_root=str(DATA_ROOT), manifest_csv=MANIFEST_CSV, tensorflow_version=tf.__version__)
(OUTPUT / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Fingerprint không phụ thuộc đường dẫn mount Kaggle; dùng so sánh các lần chạy.
import hashlib
split_records = manifest[["path", "label", "split"]].sort_values("path").to_csv(index=False, lineterminator="\n")
config["split_sha256"] = hashlib.sha256(split_records.encode("utf-8")).hexdigest()
print("Split fingerprint:", config["split_sha256"])
(OUTPUT / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")


## 2. Pipeline ảnh
Giữ tỷ lệ và padding đen về 256×256; áp dụng EXIF và chuyển RGB bằng Pillow. Chuẩn hóa nằm **trong model** để file `.keras` tự chia pixel cho 255 khi suy luận. Không chia 255 thêm lần nữa bên ngoài model.

Không shuffle validation/test để dự đoán khớp manifest. Dùng Pillow qua `tf.numpy_function` để hỗ trợ cả WebP; thuận tiện cho baseline nhưng có thể chậm hơn pipeline giải mã thuần TensorFlow.


In [ ]:
def read_rgb_numpy(path):
    path = path.item() if hasattr(path, "item") else path
    path = path.decode("utf-8") if isinstance(path, bytes) else str(path)
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert("RGB"), dtype=np.uint8)

def load_image(path, label):
    image = tf.numpy_function(read_rgb_numpy, [path], tf.uint8)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), *IMAGE_SIZE,
                                    method="bilinear", antialias=True)
    image = tf.ensure_shape(image, [*IMAGE_SIZE, 3])
    return image, tf.cast(label, tf.int32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(dtype=str), frame.label_id.to_numpy(dtype=np.int32)))
    if training:
        ds = ds.shuffle(len(frame), seed=TRAIN_SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_df = df[df.split == "train"].reset_index(drop=True)
valid_df = df[df.split == "valid"].reset_index(drop=True)
train_ds = make_dataset(train_df, training=True)
valid_ds = make_dataset(valid_df)

images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.flat:
    ax.axis("off")
for ax, im, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(np.clip(im.numpy() / 255., 0, 1))
    ax.set_title(CLASS_NAMES[int(label)])
plt.tight_layout()
plt.savefig(OUTPUT / "input_examples.png", dpi=140)
plt.show()
print("Input shape:", images.shape, "range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))

## CNN và huấn luyện
Checkpoint/early stopping đều theo val_loss. Scheduler giảm LR sau 4 epochs không cải thiện, early stopping patience12 ở improved/control. BatchNorm dùng momentum0.9; chỉ đặt Dropout sau pooling và trong classifier.

So sánh baseline_control với improved để kiểm tra gói nâng cấp kiến trúc (độ sâu + BatchNorm + Dropout). Muốn quy tác dụng cho từng thành phần cần thí nghiệm loại bỏ từng thành phần, không suy từ một lần chạy.


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(TRAIN_SEED)
L = tf.keras.layers

def build_baseline():
    return tf.keras.Sequential([
        L.Input(shape=(*IMAGE_SIZE, 3)), L.Rescaling(1./255., name="normalize_pixels"),
        L.Conv2D(32, 3, padding="same", activation="relu"), L.MaxPooling2D(2),
        L.Conv2D(64, 3, padding="same", activation="relu"), L.MaxPooling2D(2),
        L.Conv2D(128, 3, padding="same", activation="relu"), L.MaxPooling2D(2),
        L.GlobalAveragePooling2D(), L.Dense(128, activation="relu"),
        L.Dense(len(CLASS_NAMES), activation="softmax")], name="banana_cnn_baseline")

def build_improved():
    inputs = L.Input(shape=(*IMAGE_SIZE, 3))
    x = L.Rescaling(1./255., name="normalize_pixels")(inputs)
    for block, (filters, drop) in enumerate(zip([32, 64, 128, 256], [0.10, 0.15, 0.20, 0.25]), 1):
        for conv in range(1, 3):
            prefix = f"block{block}_conv{conv}"
            x = L.Conv2D(filters, 3, padding="same", use_bias=False,
                         kernel_initializer="he_normal", name=prefix)(x)
            x = L.BatchNormalization(momentum=0.9, name=prefix+"_bn")(x)
            x = L.ReLU(name=prefix+"_relu")(x)
        x = L.MaxPooling2D(2, name=f"block{block}_pool")(x)
        x = L.SpatialDropout2D(drop, name=f"block{block}_dropout")(x)
    x = L.GlobalAveragePooling2D(name="global_pool")(x)
    x = L.Dense(128, activation="relu", kernel_initializer="he_normal", name="classifier_dense")(x)
    x = L.Dropout(0.35, name="classifier_dropout")(x)
    outputs = L.Dense(len(CLASS_NAMES), activation="softmax", name="predictions")(x)
    return tf.keras.Model(inputs, outputs, name="banana_cnn_improved")

model = build_improved() if EXPERIMENT == "improved" else build_baseline()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()
# Kiểm tra forward pass với input thật trước khi fit; chưa phải đánh giá hiệu quả.
probe = model(images[:2], training=False).numpy()
assert probe.shape == (2, len(CLASS_NAMES))
assert np.isfinite(probe).all()
np.testing.assert_allclose(probe.sum(axis=1), 1., atol=1e-5)
with (OUTPUT / "model_summary.txt").open("w", encoding="utf-8") as f:
    model.summary(print_fn=lambda line, **kwargs: f.write(line + "\n"))
config["model_name"] = model.name
config["parameter_count"] = int(model.count_params())
(OUTPUT / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

callbacks = [tf.keras.callbacks.ModelCheckpoint(str(OUTPUT / "best_model.keras"),
    monitor="val_loss", mode="min", save_best_only=True, verbose=1)]
if USE_LR_SCHEDULER:
    callbacks.append(tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=4, min_delta=0., min_lr=1e-6, verbose=1))
callbacks.extend([
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=PATIENCE,
                                     restore_best_weights=True, verbose=1),
    tf.keras.callbacks.CSVLogger(str(OUTPUT / "training_log.csv")),
    tf.keras.callbacks.TerminateOnNaN(),
])
history = model.fit(train_ds, validation_data=valid_ds, epochs=EPOCHS, callbacks=callbacks)


In [ ]:
history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"
history_df.to_csv(OUTPUT / "history.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history_df.index, history_df[metric], label="Train")
    ax.plot(history_df.index, history_df["val_" + metric], label="Validation")
    ax.set(xlabel="Epoch", ylabel=metric, title=f"CNN {EXPERIMENT} — {metric}")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / "learning_curves.png", dpi=160)
plt.show()
best_epoch = int(history_df.val_loss.idxmin())
print("Best epoch theo val_loss:", best_epoch)

In [ ]:
best_model = tf.keras.models.load_model(OUTPUT / "best_model.keras")

def evaluate_split(frame, split):
    probs = best_model.predict(make_dataset(frame), verbose=1)
    y_true = frame.label_id.to_numpy()
    y_pred = probs.argmax(axis=1)
    ids = list(range(len(CLASS_NAMES)))
    metrics = {"experiment": EXPERIMENT, "train_seed": TRAIN_SEED, "split_sha256": config["split_sha256"], "parameters": int(best_model.count_params()), "split": split, "images": len(frame),
               "accuracy": float(accuracy_score(y_true, y_pred)),
               "macro_f1": float(f1_score(y_true, y_pred, labels=ids, average="macro", zero_division=0)),
               "best_epoch_by_val_loss": best_epoch}
    report = classification_report(y_true, y_pred, labels=ids, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    (OUTPUT / f"{split}_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    pd.DataFrame(report).transpose().to_csv(OUTPUT / f"{split}_classification_report.csv")
    pred = frame[["path", "label", "label_id"]].copy()
    pred["predicted_label"] = [CLASS_NAMES[i] for i in y_pred]
    pred["confidence"] = probs.max(axis=1)
    for i, name in enumerate(CLASS_NAMES):
        pred[f"prob_{name}"] = probs[:, i]
    pred.to_csv(OUTPUT / f"{split}_predictions.csv", index=False)
    cm = confusion_matrix(y_true, y_pred, labels=ids)
    pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUTPUT / f"{split}_confusion_matrix.csv")
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"CNN {EXPERIMENT} — {split} (image counts)")
    plt.tight_layout()
    plt.savefig(OUTPUT / f"{split}_confusion_matrix.png", dpi=160)
    plt.show()
    print(metrics)
    print(classification_report(y_true, y_pred, labels=ids, target_names=CLASS_NAMES, zero_division=0))
    return pred

valid_predictions = evaluate_split(valid_df, "valid")

In [ ]:
errors = valid_predictions[valid_predictions.label != valid_predictions.predicted_label]
if len(errors):
    sample = errors.sample(min(12, len(errors)), random_state=SEED)
    fig, axes = plt.subplots(int(np.ceil(len(sample) / 4)), 4, figsize=(14, 3.5 * int(np.ceil(len(sample) / 4))), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, row in zip(axes.flat, sample.itertuples()):
        im, _ = load_image(tf.constant(row.path), tf.constant(0))
        ax.imshow(np.clip(im.numpy() / 255., 0, 1))
        ax.set_title(f"True: {row.label}\nPred: {row.predicted_label} ({row.confidence:.2f})")
    plt.tight_layout()
    plt.savefig(OUTPUT / "validation_errors.png", dpi=150)
    plt.show()
else:
    print("Không có lỗi trên validation. Cần kiểm tra tính độc lập của dữ liệu trước khi kết luận.")

## 4. Suy luận ảnh riêng và test cuối cùng
Ảnh riêng dùng cùng EXIF/RGB/letterbox. Softmax confidence không phải xác suất đã được hiệu chỉnh.

Giữ `RUN_TEST=False` trong quá trình phát triển. Chỉ bật khi nhóm đã chốt mô hình; không dùng điểm test để tiếp tục chỉnh mô hình. Các file dự đoán chứa đường dẫn Kaggle; giữ ảnh/dataset tương ứng để truy xuất lại.


In [ ]:
def predict_image(path):
    image, _ = load_image(tf.constant(str(path)), tf.constant(0))
    probs = best_model.predict(image[None, ...], verbose=0)[0]
    return {"predicted_label": CLASS_NAMES[int(np.argmax(probs))],
            "probabilities": {name: float(probs[i]) for i, name in enumerate(CLASS_NAMES)}}

# predict_image("/kaggle/input/my-demo/banana.jpg")
if RUN_TEST:
    test_df = df[df.split == "test"].reset_index(drop=True)
    test_predictions = evaluate_split(test_df, "test")
else:
    print("Bỏ qua đánh giá test. Chỉ bật RUN_TEST sau khi chốt mô hình.")

import shutil
archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
print("Kết quả:", OUTPUT)
print("Tải ZIP:", archive)

## Đọc kết quả và so sánh
1. Giữ manifest và fingerprint giống nhau giữa mọi cấu hình. Để dùng manifest baseline, trỏ MANIFEST_CSV tới manifest_used.csv của lần đó.
2. Chạy improved trước. Sau đó chạy baseline_control cùng TRAIN_SEED để so sánh kiến trúc dưới cùng ngân sách huấn luyện tối đa và scheduler. Số epoch thực tế có thể khác vì early stopping.
3. So sánh validation Macro-F1, recall từng lớp, val_loss, số tham số và thời gian. Đặc biệt xem cả Semi-ripe lẫn Ripe, tránh cải thiện lớp này nhưng giảm lớp kia.
4. Sau khi chọn cấu hình hợp lý, thử TRAIN_SEED 42, 43, 44 trên cùng split và báo cáo trung bình/độ lệch chuẩn. Không chọn riêng seed tốt nhất để đại diện model.
5. Bật RUN_TEST chỉ khi đã chốt thí nghiệm. Không điều chỉnh model theo test.
6. BatchNorm/Dropout khiến train metrics lúc fit không so sánh trực tiếp hoàn toàn với inference; nếu cần train accuracy cuối, evaluate checkpoint trên train ở chế độ inference.

Không được coi bản mới tốt hơn trước khi chạy và đo. Dữ liệu/nhãn không bị notebook sửa đổi.
